# Investigating CNN CPU vs GPU Training

This notebook tests why the earlier CNN run reported a shorter wall time on CPU than on GPU. The original measurements were 5.25 s on CPU and 8.35 s on GPU for three epochs, batch size 64, and validation after each epoch. CUDA was available.

The comparison was not fully controlled: the CPU run created a fresh model, while the GPU setup moved that already-trained model to CUDA and continued training. The GPU loop also copies each mini-batch from host memory to the GPU and synchronizes when reading scalar values.

**Hypothesis:** with a small CNN and many small batches, launch, synchronization, and host-to-device transfer overhead can outweigh faster GPU arithmetic. A valid test should begin every run from identical weights, warm up each device, synchronize around timing, and report repeated-run medians. A second GPU condition preloads the dataset outside the timed region to isolate batch-transfer overhead.

In [ ]:
import gzip
import pickle
import statistics
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(42)
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print(f'GPU: {properties.name}')
    print(f'GPU memory: {properties.total_memory / 1024**3:.1f} GiB')
else:
    print('GPU condition will be skipped because CUDA is unavailable.')

## Load the Same MNIST Data
The original notebook uses `data/mnist/mnist.pkl.gz`. The path lookup below supports running this notebook from either the workspace root or the `Week_2` folder. It does not download or modify the dataset.

In [ ]:
candidate_paths = [
    Path('data/mnist/mnist.pkl.gz'),
    Path('Week_2/data/mnist/mnist.pkl.gz'),
]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find data/mnist/mnist.pkl.gz from the current directory.')

with gzip.open(data_path, 'rb') as data_file:
    ((x_train_np, y_train_np), (x_valid_np, y_valid_np), _) = pickle.load(data_file, encoding='latin-1')

x_train = torch.as_tensor(np.asarray(x_train_np), dtype=torch.float32).reshape(-1, 1, 28, 28)
y_train = torch.as_tensor(np.asarray(y_train_np), dtype=torch.long)
x_valid = torch.as_tensor(np.asarray(x_valid_np), dtype=torch.float32).reshape(-1, 1, 28, 28)
y_valid = torch.as_tensor(np.asarray(y_valid_np), dtype=torch.long)
print(f'Data file: {data_path.resolve()}')
print(f'Train: {tuple(x_train.shape)}, labels: {tuple(y_train.shape)}')
print(f'Validation: {tuple(x_valid.shape)}, labels: {tuple(y_valid.shape)}')

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.reshape(-1, 320)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

torch.manual_seed(123)
template_model = CNN()
initial_state = {name: value.detach().cpu().clone() for name, value in template_model.state_dict().items()}
loss_fn = F.cross_entropy
print(f'Model parameters: {sum(parameter.numel() for parameter in template_model.parameters()):,}')

## Controlled Timing Method
Each trial recreates the model from the same saved state and makes a new SGD optimizer. All conditions use the same data, batch size, number of epochs, and validation pass. The timed interval includes training and validation, as the original cells did.

CUDA work is asynchronous, so the timer synchronizes immediately before and after the measured region. Warm-up runs are discarded. The transfer condition moves each batch to CUDA, like the original; the resident-data condition moves the full dataset before timing, so it is a diagnostic control rather than an end-to-end data-loading measurement.

In [ ]:
def synchronize(device):
    if device.type == 'cuda':
        torch.cuda.synchronize(device)


def benchmark_trial(device_name, batch_size=64, epochs=3, resident_data=False):
    device = torch.device(device_name)
    if device.type == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA was requested but is not available.')

    model = CNN()
    model.load_state_dict(initial_state)
    model = model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.01)

    if resident_data and device.type == 'cuda':
        train_x, train_y = x_train.to(device), y_train.to(device)
        valid_x, valid_y = x_valid.to(device), y_valid.to(device)
    else:
        train_x, train_y = x_train, y_train
        valid_x, valid_y = x_valid, y_valid

    synchronize(device)
    start_time = time.perf_counter()
    validation_loss = float('nan')
    validation_accuracy = float('nan')

    for epoch in range(epochs):
        model.train()
        usable_train_size = (len(train_x) // batch_size) * batch_size
        for batch_index, start in enumerate(range(0, usable_train_size, batch_size)):
            end = start + batch_size
            x_batch = train_x[start:end]
            y_batch = train_y[start:end]
            if device.type == 'cuda' and not resident_data:
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()
            if batch_index % 10 == 0:
                _ = loss.item()

        model.eval()
        total_loss = 0.0
        total_correct = 0
        with torch.no_grad():
            for start in range(0, len(valid_x), 1000):
                end = min(start + 1000, len(valid_x))
                x_batch = valid_x[start:end]
                y_batch = valid_y[start:end]
                if device.type == 'cuda' and not resident_data:
                    x_batch = x_batch.to(device)
                    y_batch = y_batch.to(device)
                logits = model(x_batch)
                loss = loss_fn(logits, y_batch)
                predictions = logits.argmax(dim=1)
                total_loss += loss.item() * len(y_batch)
                total_correct += predictions.eq(y_batch).sum().item()
                _ = predictions.detach().cpu().numpy()
        validation_loss = total_loss / len(valid_y)
        validation_accuracy = total_correct / len(valid_y)

    synchronize(device)
    elapsed = time.perf_counter() - start_time
    return {
        'seconds': elapsed,
        'validation_loss': validation_loss,
        'validation_accuracy': validation_accuracy,
    }

## Run Repeated Trials
The defaults mirror the original three-epoch, batch-64 run. Each mode gets a one-epoch warm-up, then three timed trials. Change `REPEATS` or `EPOCHS` if you want a longer measurement; the first complete run may take a little while.

In [ ]:
modes = [('CPU', 'cpu', False)]
if torch.cuda.is_available():
    modes.extend([
        ('GPU, batch transfers', 'cuda', False),
        ('GPU, resident data', 'cuda', True),
    ])

EPOCHS = 3
REPEATS = 3
timing_results = {}

for label, device_name, resident in modes:
    print(f'Warm-up: {label}')
    benchmark_trial(device_name, batch_size=64, epochs=1, resident_data=resident)
    timing_results[label] = []
    for repeat in range(REPEATS):
        result = benchmark_trial(device_name, batch_size=64, epochs=EPOCHS, resident_data=resident)
        timing_results[label].append(result)
        print(f'  trial {repeat + 1}: {result["seconds"]:.2f}s, '
              f'val loss={result["validation_loss"]:.4f}, '
              f'val accuracy={result["validation_accuracy"]:.4f}')

summary_rows = []
for label, _, _ in modes:
    trial_times = [result['seconds'] for result in timing_results[label]]
    summary_rows.append({
        'mode': label,
        'median_seconds': statistics.median(trial_times),
        'min_seconds': min(trial_times),
        'max_seconds': max(trial_times),
        'median_validation_accuracy': statistics.median(
            result['validation_accuracy'] for result in timing_results[label]
        ),
    })

results_df = pd.DataFrame(summary_rows).set_index('mode')
display(results_df)

In [ ]:
ax = results_df['median_seconds'].plot(kind='bar', color=['#64748b', '#f97316', '#0f766e'][:len(results_df)])
ax.set_ylabel('Median wall time (seconds)')
ax.set_title('Matched CNN Training + Validation: CPU and GPU')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

cpu_seconds = results_df.loc['CPU', 'median_seconds']
print(f'Original observed ratio, GPU/CPU: {8.35 / 5.25:.2f}x')
for mode in results_df.index:
    ratio = cpu_seconds / results_df.loc[mode, 'median_seconds']
    print(f'{mode}: CPU time / mode time = {ratio:.2f}x')
if 'GPU, batch transfers' in results_df.index and 'GPU, resident data' in results_df.index:
    transfer_ratio = (
        results_df.loc['GPU, batch transfers', 'median_seconds']
        / results_df.loc['GPU, resident data', 'median_seconds']
    )
    print(f'Batch-transfer condition / resident-data condition: {transfer_ratio:.2f}x')

## Results and Interpretation

Observed in this workspace (PyTorch 2.11.0+cu128, NVIDIA RTX PRO 5000 Blackwell, 47.8 GiB):

| Mode | Median wall time | Validation accuracy |
|---|---:|---:|
| CPU | 5.20 s | 93.93% |
| GPU, batch transfers | 5.43 s | 93.95% |
| GPU, data resident before timing | 4.66 s | 93.94% |

Each result is the median of three warmed runs, each with three epochs and validation after each epoch. Every run started from the same saved weights and used the same MNIST samples. The close accuracies confirm that the controlled runs did comparable work.

The transfer-mode GPU run was about 4% slower than CPU, while keeping data on the GPU before timing was about 11% faster than CPU. The batch-transfer condition was about 16% slower than the resident-data condition. This supports the overhead explanation: for this small CNN and batch size, moving data and synchronizing can erase the GPU's compute advantage. It does not prove transfers are the only cause; the resident-data timing intentionally excludes the initial data copy.

The original single runs were 5.25 s CPU and 8.35 s GPU, a 1.59x GPU/CPU ratio. Those are less reliable than the repeated synchronized measurements, and their model states differed: the CPU run started fresh, but the GPU run continued from the CPU-trained model. That also explains why the old accuracy values differed substantially.

The measured result is specific to this machine, dataset, model, and batch size. Larger models or batches can shift the balance toward GPU. The cells above let you rerun the experiment and inspect your own timings.